# Ablation Repair Notebook (final-protocol aligned)

This notebook regenerates **Table 4 ablation results** under a clean, single-run workflow aligned with the final in-domain protocol.

It does **not** overwrite previous in-domain, repro-fix, or improvement outputs.

**What it does**
- rebuilds the Memotion split with the same fixed seed used in the main runs
- trains lightweight ablation variants on the same split
- pulls the **proposed full model** row directly from the frozen **v4 final run**
- exports a synchronized ablation table and per-variant summaries into a dedicated folder

Run this notebook **top-to-bottom**.


In [1]:
from pathlib import Path

# Core paths
DATA_ROOT = Path(r"D:\AI\datasets\Memotion\memotion_dataset_7k")
HF_CACHE = Path(r"D:\AI\hf_cache")
OUTPUT_ROOT = Path(r"D:\AI\outputs\runs_sarcasm_ablation\ablation_repair_v1_finalprotocol")
V4_ROOT = Path(r"D:\AI\outputs\runs_sarcasm_improve")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT   =", DATA_ROOT)
print("HF_CACHE    =", HF_CACHE)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("V4_ROOT     =", V4_ROOT)

DATA_ROOT   = D:\AI\datasets\Memotion\memotion_dataset_7k
HF_CACHE    = D:\AI\hf_cache
OUTPUT_ROOT = D:\AI\outputs\runs_sarcasm_ablation\ablation_repair_v1_finalprotocol
V4_ROOT     = D:\AI\outputs\runs_sarcasm_improve


In [2]:
import os, json, math, random, warnings, copy, glob, re
from pathlib import Path

os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("TRANSFORMERS_CACHE", str(HF_CACHE / "transformers"))
HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN set:", HF_TOKEN is not None)

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)


HF_TOKEN set: False
device = cuda


In [3]:
# Locate labels file and image directory
label_candidates = []
for pat in ["labels*.csv", "labels*.xlsx", "labels*.xls", "**/labels*.csv", "**/labels*.xlsx", "**/labels*.xls"]:
    label_candidates += glob.glob(str(DATA_ROOT / pat), recursive=True)
label_candidates = sorted(label_candidates, key=lambda p: (0 if p.lower().endswith(".csv") else 1, len(p)))
CSV_PATH = Path(label_candidates[0]) if label_candidates else None

img_exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
IMAGES_DIR = None
best_count = 0
for root, dirs, files in os.walk(DATA_ROOT):
    c = sum(1 for f in files if f.lower().endswith(img_exts))
    if c > best_count:
        best_count = c
        IMAGES_DIR = Path(root)

print("CSV_PATH   =", CSV_PATH)
print("IMAGES_DIR =", IMAGES_DIR, f"(images: {best_count})")
if CSV_PATH is None or IMAGES_DIR is None:
    raise FileNotFoundError("Could not locate labels file or images directory under DATA_ROOT")


CSV_PATH   = D:\AI\datasets\Memotion\memotion_dataset_7k\labels.csv
IMAGES_DIR = D:\AI\datasets\Memotion\memotion_dataset_7k\images (images: 6991)


In [4]:
# Load labels and reconstruct the same binary task used in the main runs
if CSV_PATH.suffix.lower() == ".csv":
    df0 = pd.read_csv(CSV_PATH)
else:
    df0 = pd.read_excel(CSV_PATH)

print("Columns:", list(df0.columns))

# infer text/image/label columns using the same conventions as the main notebook
IMG_COL_CANDIDATES = ["image_name", "image", "img", "file_name", "filename"]
TEXT_COL_CANDIDATES = ["text_corrected", "text_ocr", "text", "ocr_text"]
LABEL_COL_CANDIDATES = ["sarcasm", "label", "sarcasm_label"]

IMG_COL = next((c for c in IMG_COL_CANDIDATES if c in df0.columns), None)
TEXT_COL = next((c for c in TEXT_COL_CANDIDATES if c in df0.columns), None)
RAW_LABEL_COL = next((c for c in LABEL_COL_CANDIDATES if c in df0.columns), None)

if IMG_COL is None or TEXT_COL is None or RAW_LABEL_COL is None:
    raise RuntimeError(f"Could not infer IMG_COL/TEXT_COL/RAW_LABEL_COL. Found IMG={IMG_COL}, TEXT={TEXT_COL}, LABEL={RAW_LABEL_COL}")

sar_map = {
    "not_sarcastic": 0,
    "general": 1,
    "twisted_meaning": 1,
    "very_twisted": 1,
    0: 0,
    1: 1,
}

memotion_df = df0[[IMG_COL, TEXT_COL, RAW_LABEL_COL]].copy()
memotion_df.columns = ["image_name", "text", "sarcasm_raw"]
memotion_df = memotion_df.dropna(subset=["image_name", "text", "sarcasm_raw"]).copy()
memotion_df["image_name"] = memotion_df["image_name"].astype(str)
memotion_df["text"] = memotion_df["text"].astype(str)
memotion_df["binary_label"] = memotion_df["sarcasm_raw"].map(sar_map)
memotion_df = memotion_df[memotion_df["binary_label"].isin([0,1])].copy()
memotion_df["binary_label"] = memotion_df["binary_label"].astype(int)

print("Memotion usable rows:", len(memotion_df))
print(memotion_df["binary_label"].value_counts(dropna=False))


Columns: ['Unnamed: 0', 'image_name', 'text_ocr', 'text_corrected', 'humour', 'sarcasm', 'offensive', 'motivational', 'overall_sentiment']
Memotion usable rows: 6987
binary_label
1    5444
0    1543
Name: count, dtype: int64


In [5]:
# Rebuild the same split structure used in the main in-domain runs
train_df, temp_df = train_test_split(
    memotion_df,
    test_size=0.30,
    random_state=SEED,
    stratify=memotion_df["binary_label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["binary_label"]
)

print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)
print("Test positive fraction:", float(test_df["binary_label"].mean()))


Train/Val/Test: (4890, 4) (1048, 4) (1049, 4)
Test positive fraction: 0.778836987607245


In [6]:
# Emoji helper
EMOJI_RE = re.compile("[𐀀-􏿿]", flags=re.UNICODE)

def extract_emojis(s):
    if not isinstance(s, str):
        return []
    return EMOJI_RE.findall(s)

emoji_set = set()
for txt in tqdm(train_df["text"].tolist(), desc="Building emoji vocab"):
    emoji_set.update(extract_emojis(txt))
emoji2id = {e:i+1 for i,e in enumerate(sorted(emoji_set))}
print("emoji vocab size:", len(emoji2id))

def emojis_to_ids(text, max_emojis=16):
    ids = [emoji2id.get(e, 0) for e in extract_emojis(text)[:max_emojis]]
    ids += [0] * (max_emojis - len(ids))
    return ids


Building emoji vocab:   0%|          | 0/4890 [00:00<?, ?it/s]

emoji vocab size: 1


In [7]:
# Tokenizer and transforms
TEXT_MODEL = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, token=HF_TOKEN, use_fast=False)

resnet_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

clip_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466,0.4578275,0.40821073],
                         std=[0.26862954,0.26130258,0.27577711]),
])


In [8]:
class MemeDataset(Dataset):
    def __init__(self, df, images_dir, text_col="text", max_len=64, max_emojis=16):
        self.df = df.reset_index(drop=True)
        self.images_dir = str(images_dir)
        self.text_col = text_col
        self.max_len = max_len
        self.max_emojis = max_emojis

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, str(r["image_name"]))
        y = int(r["binary_label"])
        text = str(r[self.text_col])

        ok = 1
        try:
            img = Image.open(img_path).convert("RGB")
            pixel_resnet = resnet_transform(img)
            pixel_clip = clip_transform(img)
        except Exception:
            pixel_resnet = torch.zeros(3,224,224)
            pixel_clip = torch.zeros(3,224,224)
            ok = 0

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        eids = torch.tensor(emojis_to_ids(text, self.max_emojis), dtype=torch.long)
        return {
            "pixel_resnet": pixel_resnet,
            "pixel_clip": pixel_clip,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "emoji_ids": eids,
            "label": torch.tensor(y, dtype=torch.float32),
            "ok": torch.tensor(ok, dtype=torch.long)
        }

def collate_fn(batch):
    batch = [b for b in batch if int(b["ok"]) == 1]
    if len(batch) == 0:
        return None
    out = {}
    for k in ["pixel_resnet","pixel_clip","input_ids","attention_mask","emoji_ids","label"]:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out

train_ds = MemeDataset(train_df, IMAGES_DIR)
val_ds = MemeDataset(val_df, IMAGES_DIR)
test_ds = MemeDataset(test_df, IMAGES_DIR)
print("datasets ready")


datasets ready


In [9]:
# Final-protocol loaders for ablations: sampler ON, batch size 8
BATCH_SIZE = 8
NUM_WORKERS = 0

y_tr = train_df["binary_label"].astype(int).values
class_counts = np.bincount(y_tr, minlength=2)
class_w = 1.0 / np.maximum(class_counts, 1)
sample_w = class_w[y_tr].astype(np.float32)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_w),
    num_samples=len(sample_w),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False,
                        collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False,
                         collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print("class_counts:", class_counts, "| sampler ON")


class_counts: [1080 3810] | sampler ON


In [10]:
def safe_auc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true)
        y_prob = np.asarray(y_prob)
        if len(np.unique(y_true)) < 2:
            return 0.0
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return 0.0

def tune_threshold(y_true, y_prob):
    best_thr, best_f1 = 0.50, -1.0
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    for thr in np.arange(0.05, 0.96, 0.05):
        pred = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, pred, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def batch_to_device(batch, dev):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(dev) if torch.is_tensor(v) else v
    return out

def get_max_ids_from_loader(loader):
    max_token_id = 0
    max_emoji_id = 0
    for b in loader:
        if b is None:
            continue
        max_token_id = max(max_token_id, int(b["input_ids"].max().item()))
        max_emoji_id = max(max_emoji_id, int(b["emoji_ids"].max().item()))
    return max_token_id, max_emoji_id

def evaluate_model(model, loader, thr=0.5):
    model.eval()
    ys, probs = [], []
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            batch = batch_to_device(batch, device)
            y = batch["label"].float().view(-1).cpu().numpy().astype(int)
            out = model(batch)
            logits = out["logits"].view(-1)
            p = torch.sigmoid(logits).detach().cpu().numpy()
            ys.extend(y.tolist())
            probs.extend(p.tolist())
    ys = np.asarray(ys).astype(int)
    probs = np.asarray(probs).astype(float)
    pred = (probs >= thr).astype(int)
    return {
        "acc": float(accuracy_score(ys, pred)),
        "macro_f1": float(f1_score(ys, pred, average="macro", zero_division=0)),
        "auc": safe_auc(ys, probs),
        "y_true": ys,
        "y_prob": probs,
        "y_pred": pred,
    }


In [11]:
# Model definitions for ablation variants
class TextBiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, pad_idx=0, dropout=0.3):
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.embedding = nn.Embedding(self.vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = hidden_dim * 2
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        out, _ = self.lstm(x)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            out = out * mask
            denom = mask.sum(dim=1).clamp(min=1.0)
            pooled = out.sum(dim=1) / denom
        else:
            pooled = out.mean(dim=1)
        return self.dropout(pooled)

class TextCNNEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, num_filters=128, kernels=(3,4,5), pad_idx=0, dropout=0.3):
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.embedding = nn.Embedding(self.vocab_size, emb_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, num_filters, k) for k in kernels])
        self.dropout = nn.Dropout(dropout)
        self.out_dim = num_filters * len(kernels)
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids).transpose(1, 2)
        feats = []
        for conv in self.convs:
            z = F.relu(conv(x))
            z = F.max_pool1d(z, kernel_size=z.size(2)).squeeze(2)
            feats.append(z)
        return self.dropout(torch.cat(feats, dim=1))

class ResNet50Encoder(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        backbone = models.resnet50(weights=None)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.out_dim = in_features
    def forward(self, pixel_resnet):
        z = self.backbone(pixel_resnet)
        return self.dropout(z)

class SmallClipLikeEncoder(nn.Module):
    def __init__(self, out_dim=512, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(128, out_dim)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = out_dim
    def forward(self, pixel_clip):
        x = self.cnn(pixel_clip).flatten(1)
        x = self.fc(x)
        return self.dropout(x)

class DLTextOnlyBiLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.fc = nn.Linear(self.text.out_dim, 1)
    def forward(self, batch):
        z = self.text(batch["input_ids"], batch["attention_mask"])
        return {"logits": self.fc(z).squeeze(-1)}

class DLTextOnlyCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextCNNEncoder(vocab_size)
        self.fc = nn.Linear(self.text.out_dim, 1)
    def forward(self, batch):
        z = self.text(batch["input_ids"], batch["attention_mask"])
        return {"logits": self.fc(z).squeeze(-1)}

class DLImageOnlyResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        self.image = ResNet50Encoder()
        self.fc = nn.Linear(self.image.out_dim, 1)
    def forward(self, batch):
        z = self.image(batch["pixel_resnet"])
        return {"logits": self.fc(z).squeeze(-1)}

class DLTextImageConcat(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.image = ResNet50Encoder()
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + self.image.out_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zi = self.image(batch["pixel_resnet"])
        return {"logits": self.fc(torch.cat([zt, zi], dim=1)).squeeze(-1)}

class DLTextImageClipFusion(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.resnet = ResNet50Encoder()
        self.clip_like = SmallClipLikeEncoder(out_dim=512)
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + self.resnet.out_dim + self.clip_like.out_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zr = self.resnet(batch["pixel_resnet"])
        zc = self.clip_like(batch["pixel_clip"])
        return {"logits": self.fc(torch.cat([zt, zr, zc], dim=1)).squeeze(-1)}

class HybridConcatEmoji(nn.Module):
    def __init__(self, vocab_size, emoji_vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.image = ResNet50Encoder()
        self.emoji_emb = nn.Embedding(int(emoji_vocab_size), 32, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + self.image.out_dim + 32, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zi = self.image(batch["pixel_resnet"])
        ze = self.emoji_emb(batch["emoji_ids"]).mean(dim=1)
        return {"logits": self.fc(torch.cat([zt, zi, ze], dim=1)).squeeze(-1)}

class HybridCrossAttnNoGWO(nn.Module):
    def __init__(self, vocab_size, emoji_vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.image = ResNet50Encoder()
        self.emoji_emb = nn.Embedding(int(emoji_vocab_size), 32, padding_idx=0)
        self.q_text = nn.Linear(self.text.out_dim, 128)
        self.k_img = nn.Linear(self.image.out_dim, 128)
        self.v_img = nn.Linear(self.image.out_dim, 128)
        self.q_emoji = nn.Linear(32, 128)
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + 128 + 128, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zi = self.image(batch["pixel_resnet"])
        ze = self.emoji_emb(batch["emoji_ids"]).mean(dim=1)
        qt = self.q_text(zt)
        ki = self.k_img(zi)
        vi = self.v_img(zi)
        qe = self.q_emoji(ze)
        attn_ti = torch.sigmoid((qt * ki).sum(dim=1, keepdim=True) / math.sqrt(128))
        fused_ti = attn_ti * vi
        attn_te = torch.sigmoid((qt * qe).sum(dim=1, keepdim=True) / math.sqrt(128))
        fused_te = attn_te * qe
        return {"logits": self.fc(torch.cat([zt, fused_ti, fused_te], dim=1)).squeeze(-1)}


In [12]:
# Safe vocab sizes inferred from the actual split
train_tok_max, train_emo_max = get_max_ids_from_loader(train_loader)
val_tok_max, val_emo_max = get_max_ids_from_loader(val_loader)
test_tok_max, test_emo_max = get_max_ids_from_loader(test_loader)

global_token_max = max(train_tok_max, val_tok_max, test_tok_max)
global_emoji_max = max(train_emo_max, val_emo_max, test_emo_max)

vocab_size = int(global_token_max + 33)
emoji_vocab_size = int(global_emoji_max + 33)
print("vocab_size =", vocab_size)
print("emoji_vocab_size =", emoji_vocab_size)


vocab_size = 199997
emoji_vocab_size = 33


In [13]:
# Training helper for ablation models
EPOCHS_ABL = 4

def train_one_model(model_name, model, train_loader, val_loader, test_loader, epochs=EPOCHS_ABL, lr=1e-3):
    model_dir = OUTPUT_ROOT / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    best_state = None
    best_val_f1 = -1.0
    best_thr = 0.50
    hist = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for batch in train_loader:
            if batch is None:
                continue
            batch = batch_to_device(batch, device)
            y = batch["label"].float().view(-1)
            optimizer.zero_grad()
            logits = model(batch)["logits"].view(-1)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        val_eval = evaluate_model(model, val_loader, thr=0.50)
        thr, tuned_f1 = tune_threshold(val_eval["y_true"], val_eval["y_prob"])
        hist.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)) if losses else 0.0,
            "val_acc": val_eval["acc"],
            "val_macro_f1_at_0_5": val_eval["macro_f1"],
            "val_auc": val_eval["auc"],
            "best_thr_now": thr,
            "best_val_macro_f1_now": tuned_f1
        })
        print(f"{model_name} | epoch {epoch}: loss={hist[-1]['train_loss']:.4f}, val_f1={tuned_f1:.4f}, thr={thr:.2f}")
        if tuned_f1 > best_val_f1:
            best_val_f1 = tuned_f1
            best_thr = thr
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    test_eval = evaluate_model(model, test_loader, thr=best_thr)
    pd.DataFrame(hist).to_csv(model_dir / "history.csv", index=False)
    pd.DataFrame({
        "y_true": test_eval["y_true"],
        "y_prob": test_eval["y_prob"],
        "y_pred": test_eval["y_pred"]
    }).to_csv(model_dir / "predictions.csv", index=False)

    summary = {
        "model": model_name,
        "status": "ok",
        "error": "",
        "params_trainable": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "best_val_macro_f1": float(best_val_f1),
        "best_thr": float(best_thr),
        "test_acc": float(test_eval["acc"]),
        "test_macro_f1": float(test_eval["macro_f1"]),
        "test_auc": float(test_eval["auc"])
    }
    with open(model_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    return summary


In [14]:
# Pull proposed full model row directly from frozen v4 final run
v4_candidates = sorted(V4_ROOT.glob("improve_v4*"))
if not v4_candidates:
    raise FileNotFoundError(f"Could not find improve_v4 run folder under {V4_ROOT}")
V4_RUN = v4_candidates[0]
FINAL_EVAL_V4 = V4_RUN / "final_eval_fixed.json"
if not FINAL_EVAL_V4.exists():
    raise FileNotFoundError(f"Missing final_eval_fixed.json in {V4_RUN}")

v4_eval = json.loads(FINAL_EVAL_V4.read_text(encoding="utf-8"))
proposed_row = {
    "model": "proposed_full_model",
    "status": "ok",
    "error": "",
    "params_trainable": np.nan,
    "best_val_macro_f1": float(v4_eval.get("multi_thr_best", {}).get("macro_f1", np.nan)),
    "best_thr": float(v4_eval.get("thresholds", {}).get("val_multi_thr", np.nan)),
    "test_acc": float(v4_eval.get("multi_thr_best", {}).get("acc", np.nan)),
    "test_macro_f1": float(v4_eval.get("multi_thr_best", {}).get("macro_f1", np.nan)),
    "test_auc": float(v4_eval.get("multi_thr_best", {}).get("auc", np.nan)),
    "source": str(V4_RUN)
}
print(json.dumps(proposed_row, indent=2))


{
  "model": "proposed_full_model",
  "status": "ok",
  "error": "",
  "params_trainable": NaN,
  "best_val_macro_f1": 0.5059948070461256,
  "best_thr": 0.525,
  "test_acc": 0.6472831267874166,
  "test_macro_f1": 0.5059948070461256,
  "test_auc": 0.5032996750600662,
  "source": "D:\\AI\\outputs\\runs_sarcasm_improve\\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER"
}


In [15]:
# Train ablation variants under the same split / sampler-on protocol
model_specs = [
    ("text_only_bilstm",      DLTextOnlyBiLSTM(vocab_size),                       4, 1e-3),
    ("text_only_textcnn",     DLTextOnlyCNN(vocab_size),                          4, 1e-3),
    ("image_only_resnet50",   DLImageOnlyResNet50(),                              4, 1e-4),
    ("text_image_concat",     DLTextImageConcat(vocab_size),                      4, 1e-4),
    ("text_image_clipfusion", DLTextImageClipFusion(vocab_size),                  4, 1e-4),
    ("hybrid_concat_emoji",   HybridConcatEmoji(vocab_size, emoji_vocab_size),    4, 1e-4),
    ("hybrid_crossattn_no_gwo", HybridCrossAttnNoGWO(vocab_size, emoji_vocab_size), 4, 1e-4),
]

rows = [proposed_row]
for model_name, model_obj, epochs, lr in model_specs:
    try:
        row = train_one_model(model_name, model_obj, train_loader, val_loader, test_loader, epochs=epochs, lr=lr)
    except Exception as e:
        print(f"[FAILED] {model_name}: {type(e).__name__}: {e}")
        row = {
            "model": model_name,
            "status": "failed",
            "error": f"{type(e).__name__}: {e}",
            "params_trainable": 0,
            "best_val_macro_f1": 0.0,
            "best_thr": 0.5,
            "test_acc": 0.0,
            "test_macro_f1": 0.0,
            "test_auc": 0.0,
        }
    rows.append(row)

ablation_df = pd.DataFrame(rows)
ablation_df


text_only_bilstm | epoch 1: loss=0.6753, val_f1=0.5186, thr=0.40
text_only_bilstm | epoch 2: loss=0.5442, val_f1=0.5254, thr=0.50
text_only_bilstm | epoch 3: loss=0.3581, val_f1=0.5487, thr=0.60
text_only_bilstm | epoch 4: loss=0.2036, val_f1=0.5365, thr=0.65
text_only_textcnn | epoch 1: loss=0.6124, val_f1=0.5192, thr=0.50
text_only_textcnn | epoch 2: loss=0.4369, val_f1=0.5284, thr=0.55
text_only_textcnn | epoch 3: loss=0.2851, val_f1=0.5140, thr=0.85
text_only_textcnn | epoch 4: loss=0.2106, val_f1=0.5096, thr=0.90
image_only_resnet50 | epoch 1: loss=0.7358, val_f1=0.5101, thr=0.30
image_only_resnet50 | epoch 2: loss=0.7280, val_f1=0.5172, thr=0.40
image_only_resnet50 | epoch 3: loss=0.7131, val_f1=0.5050, thr=0.40
image_only_resnet50 | epoch 4: loss=0.7144, val_f1=0.5019, thr=0.55
text_image_concat | epoch 1: loss=0.7083, val_f1=0.5232, thr=0.50
text_image_concat | epoch 2: loss=0.6946, val_f1=0.4613, thr=0.45
text_image_concat | epoch 3: loss=0.6945, val_f1=0.4997, thr=0.45
text_i

,model,status,error,params_trainable,best_val_macro_f1,best_thr,test_acc,test_macro_f1,test_auc,source
0,proposed_full_model,ok,,NaN,0.505995,0.525,0.647283,0.505995,0.503300,D:\AI\outputs\runs_sarcasm_improve\improve_v4_...
1,text_only_bilstm,ok,,25864065.0,0.548697,0.600,0.670162,0.502476,0.521549,NaN
2,text_only_textcnn,ok,,25796993.0,0.528407,0.550,0.658723,0.509198,0.510691,NaN
3,image_only_resnet50,ok,,23510081.0,0.517236,0.400,0.695901,0.506372,0.474945,NaN
4,text_image_concat,ok,,49962177.0,0.530783,0.450,0.673022,0.490371,0.462763,NaN
5,text_image_clipfusion,ok,,50252545.0,0.504582,0.500,0.633937,0.487303,0.470389,NaN
6,hybrid_concat_emoji,ok,,49971425.0,0.529693,0.450,0.669209,0.508370,0.477694,NaN
7,hybrid_crossattn_no_gwo,ok,,50066145.0,0.511377,0.550,0.606292,0.464347,0.462837,NaN


In [16]:
# Export final repaired Table 4 artifacts
summary_path = OUTPUT_ROOT / "ablation_summary_repaired.csv"
table4_path = OUTPUT_ROOT / "table4_ablation_final.csv"
manifest_path = OUTPUT_ROOT / "artifact_manifest_ablation_repair.json"

ablation_df.to_csv(summary_path, index=False)

# Manuscript-friendly display names and delta vs proposed full model
name_map = {
    "proposed_full_model": "Full model (Primary, p multi)",
    "text_only_textcnn": "Text-only (TextCNN)",
    "text_only_bilstm": "Text-only (BiLSTM)",
    "image_only_resnet50": "Image-only (ResNet-50)",
    "text_image_concat": "Text+Image (Concat)",
    "text_image_clipfusion": "Text+Image (CLIP fusion)",
    "hybrid_concat_emoji": "Hybrid (Concat + Emoji)",
    "hybrid_crossattn_no_gwo": "Hybrid (Cross-attn, no GWO)",
}

table4 = ablation_df.copy()
table4["Variant"] = table4["model"].map(name_map).fillna(table4["model"])
base_f1 = float(table4.loc[table4["model"] == "proposed_full_model", "test_macro_f1"].iloc[0])
table4["Acc"] = table4["test_acc"].round(3)
table4["Macro-F1"] = table4["test_macro_f1"].round(3)
table4["ROC-AUC"] = table4["test_auc"].round(3)
table4["ΔMacro-F1"] = (table4["test_macro_f1"] - base_f1).round(3)
table4 = table4[["Variant","Acc","Macro-F1","ROC-AUC","ΔMacro-F1","status","error"]]
table4.to_csv(table4_path, index=False)

manifest = {
    "output_root": str(OUTPUT_ROOT),
    "v4_source_run": str(V4_RUN),
    "generated_files": sorted([p.name for p in OUTPUT_ROOT.iterdir() if p.is_file()]),
    "subdirs": sorted([p.name for p in OUTPUT_ROOT.iterdir() if p.is_dir()]),
    "note": "Ablation table regenerated under final split/sampler-on protocol; proposed full model row sourced from frozen v4 final run."
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print("Saved:", table4_path)
print("Saved:", manifest_path)
display(table4)


Saved: D:\AI\outputs\runs_sarcasm_ablation\ablation_repair_v1_finalprotocol\ablation_summary_repaired.csv
Saved: D:\AI\outputs\runs_sarcasm_ablation\ablation_repair_v1_finalprotocol\table4_ablation_final.csv
Saved: D:\AI\outputs\runs_sarcasm_ablation\ablation_repair_v1_finalprotocol\artifact_manifest_ablation_repair.json


,Variant,Acc,Macro-F1,ROC-AUC,ΔMacro-F1,status,error
0,"Full model (Primary, p multi)",0.647,0.506,0.503,0.000,ok,
1,Text-only (BiLSTM),0.670,0.502,0.522,-0.004,ok,
2,Text-only (TextCNN),0.659,0.509,0.511,0.003,ok,
3,Image-only (ResNet-50),0.696,0.506,0.475,0.000,ok,
4,Text+Image (Concat),0.673,0.490,0.463,-0.016,ok,
5,Text+Image (CLIP fusion),0.634,0.487,0.470,-0.019,ok,
6,Hybrid (Concat + Emoji),0.669,0.508,0.478,0.002,ok,
7,"Hybrid (Cross-attn, no GWO)",0.606,0.464,0.463,-0.042,ok,
